# Session 3 · Association Rule Mining

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/01-data-mining-with-weka/code/session-03-association-rule-mining.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

### Example 1 · Support, confidence and lift by hand in Python

In [1]:
baskets = [
    {"bread", "milk"},
    {"bread", "butter", "milk"},
    {"bread", "butter"},
    {"milk", "eggs"},
    {"bread", "butter", "milk", "eggs"},
]
N = len(baskets)

def support(items):
    return sum(items <= b for b in baskets) / N

X, Y = {"bread", "butter"}, {"milk"}
s_xy = support(X | Y)
conf = s_xy / support(X)
lift = conf / support(Y)
print(f"support(X)      = {support(X):.2f}")
print(f"support(X u Y)  = {s_xy:.2f}")
print(f"confidence      = {conf:.2f}")
print(f"lift            = {lift:.2f}")

support(X)      = 0.60
support(X u Y)  = 0.40
confidence      = 0.67
lift            = 0.83


### Example 5 · Apriori in Python with mlxtend

In [2]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

transactions = [
    ["bread", "butter", "milk"], ["bread", "butter", "jam"], ["bread", "milk", "tea"],
    ["milk", "eggs", "tea"], ["bread", "butter", "milk", "jam"], ["bread", "butter", "jam", "tea"],
    ["milk", "eggs"], ["bread", "butter", "milk", "eggs"], ["tea"], ["bread", "butter", "jam"],
]
te = TransactionEncoder()
onehot = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)

frequent = apriori(onehot, min_support=0.3, use_colnames=True)
print("Frequent itemsets (support >= 0.3):")
show = frequent.copy()  # keep the frozensets intact for association_rules
show["itemsets"] = show["itemsets"].apply(lambda s: ", ".join(sorted(s)))
print(show.sort_values("support", ascending=False).to_string(index=False))

rules = association_rules(frequent, metric="confidence", min_threshold=0.8)
cols = ["antecedents", "consequents", "support", "confidence", "lift"]
rules = rules[cols].sort_values("lift", ascending=False)
rules["antecedents"] = rules["antecedents"].apply(lambda s: ", ".join(sorted(s)))
rules["consequents"] = rules["consequents"].apply(lambda s: ", ".join(sorted(s)))
print("\nRules (confidence >= 0.8):")
print(rules.round(2).to_string(index=False))

Frequent itemsets (support >= 0.3):
 support            itemsets
     0.7               bread
     0.6              butter
     0.6                milk
     0.6       bread, butter
     0.4                 jam
     0.4                 tea
     0.4          bread, jam
     0.4         bread, milk
     0.4         butter, jam
     0.4  bread, butter, jam
     0.3                eggs
     0.3        butter, milk
     0.3          eggs, milk
     0.3 bread, butter, milk

Rules (confidence >= 0.8):
 antecedents   consequents  support  confidence  lift
         jam        butter      0.4        1.00  1.67
        eggs          milk      0.3        1.00  1.67
  bread, jam        butter      0.4        1.00  1.67
         jam bread, butter      0.4        1.00  1.67
       bread        butter      0.6        0.86  1.43
      butter         bread      0.6        1.00  1.43
         jam         bread      0.4        1.00  1.43
 butter, jam         bread      0.4        1.00  1.43
butter, milk   

### Example 6 · How min_support changes the result

In [3]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

transactions = [
    ["bread", "butter", "milk"], ["bread", "butter", "jam"], ["bread", "milk", "tea"],
    ["milk", "eggs", "tea"], ["bread", "butter", "milk", "jam"], ["bread", "butter", "jam", "tea"],
    ["milk", "eggs"], ["bread", "butter", "milk", "eggs"], ["tea"], ["bread", "butter", "jam"],
]
te = TransactionEncoder()
onehot = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)

print("min_support  itemsets  rules(conf>=0.6)")
for ms in [0.1, 0.2, 0.3, 0.4, 0.5]:
    fi = apriori(onehot, min_support=ms, use_colnames=True)
    r = association_rules(fi, metric="confidence", min_threshold=0.6) if len(fi) else []
    print(f"{ms:>10}  {len(fi):>8}  {len(r):>6}")

min_support  itemsets  rules(conf>=0.6)
       0.1        35      38
       0.2        16      14
       0.3        14      14
       0.4        10      11
       0.5         4       2
